In [ ]:
import os
import glob
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from dotenv import load_dotenv
from models.db_models import TrainingDataGenMetaData, StoryverseMetaData, TrainingIOPair
from llm_util import OpenAiLLMProvider, LLMProvider
from db_utils import DatabaseManager

load_dotenv()

In [ ]:
def extract_response_text(response) -> str:
    """Extract text from LLM response output"""
    if not response or not response.output:
        raise ValueError("Invalid response from LLM provider")
    
    for output in response.output:
        if output.type == "message" and output.content and len(output.content) > 0:
            return output.content[0].text
    
    return ""

In [ ]:
# Constants

title_examples = "'A Study in Scarlet', 'The Sign of the Four', 'A Scandal in Bohemia', 'The Red-Headed League', 'A Case of Identity', 'The Adventure of the Blue Carbuncle', 'The Adventure of the Speckled Band', 'The Adventure of the Copper Beeches', 'The Boscombe Valley Mystery', 'The Five Orange Pips', 'The Man with the Twisted Lip', 'The Blue Carbuncle', 'The Speckled Band', 'The Engineer's Thumb', 'The Adventure of the Engineer's Thumb', 'The Adventure of the Noble Bachelor', 'The Adventure of the Beryl Coronet', 'The Noble Bachelor', 'The Beryl Coronet', 'The Copper Beeches', 'Silver Blaze', 'The Yellow Face', 'The Stockbroker's Clerk', 'The Gloria Scott', 'The Musgrave Ritual', 'The Reigate Squires', 'The Adventure of the Greek Interpreter', 'The Adventure of the Naval Treaty', 'The Crooked Man', 'The Resident Patient', 'The Greek Interpreter', 'The Naval Treaty', 'The Adventure of the Final Problem', 'The Final Problem', 'The Empty House', 'The Norwood Builder', 'The Dancing Men', 'The Adventure of the Empty House', 'The Adventure of the Norwood Builder', 'The Solitary Cyclist', 'The Priory School"

In [ ]:
def generateTrainingIOPair(
    first_draft_file_path: str,
    training_meta_data: TrainingDataGenMetaData,
    storyverse_meta_data: StoryverseMetaData,
    llm_provider: LLMProvider,
    db_manager: DatabaseManager,
    story_verse: str = "SHERLOCK"
) -> dict:
    """Generate training IO pairs for all pipeline steps from a first draft story"""
    
    print(f"\nProcessing file: {first_draft_file_path}")
    
    # Extract story title from filename (without extension)
    story_title = os.path.splitext(os.path.basename(first_draft_file_path))[0]
    
    # Read the first draft
    with open(first_draft_file_path, 'r', encoding='utf-8') as f:
        first_draft = f.read()
    
    # Get current epoch time
    created_at = int(time.time())
    
    # Store the generated outputs for reference in later steps
    generated_outputs = {}
    created_pairs = []
    
    # STEP 1: CHARACTER_DATA_GEN
    print("  - Generating CHARACTER_DATA_GEN pair...")
    char_pair = TrainingIOPair(
        storyVerse=story_verse,
        storyTitle=story_title,
        pipelineStepName="CHARACTER_DATA_GEN",
        systemPrompt=training_meta_data.storyVerseSystemPrompt,
        reverseOutputPrompt="",
        inputPromptForFineTuning="",
        modelOutput="",
        createdAt=created_at
    )
    
    # Generate model output using reverse prompt
    prompt = f"{training_meta_data.characterGenearationReversePrompt}\n\n{first_draft}"
    char_pair.reverseOutputPrompt = prompt  # Save the actual prompt sent to LLM
    response_id = llm_provider.initiateResponse(
        prompt,
        model="gpt-4.1-2025-04-14",
        resoning_effort="medium"
    )
    response = llm_provider.getPooledResponse(response_id)
    char_pair.modelOutput = extract_response_text(response)
    generated_outputs["CHARACTER_DATA_GEN"] = char_pair.modelOutput
    
    # Generate input prompt for fine-tuning
    prompt = f"{training_meta_data.characterInputPomptGenrationPrompt}\n\n{char_pair.modelOutput}"
    response_id = llm_provider.initiateResponse(
        prompt,
        model="gpt-4.1-2025-04-14",
        resoning_effort="medium"
    )
    response = llm_provider.getPooledResponse(response_id)
    char_pair.inputPromptForFineTuning = extract_response_text(response)
    
    # Save to DB
    char_id = db_manager.create_training_io_pair(char_pair)
    created_pairs.append(("CHARACTER_DATA_GEN", char_id))
    print(f"    ✓ Saved CHARACTER_DATA_GEN pair with ID: {char_id}")
    
    # STEP 2: PLOT_GEN
    print("  - Generating PLOT_GEN pair...")
    plot_pair = TrainingIOPair(
        storyVerse=story_verse,
        storyTitle=story_title,
        pipelineStepName="PLOT_GEN",
        systemPrompt=training_meta_data.storyVerseSystemPrompt,
        reverseOutputPrompt="",
        inputPromptForFineTuning="",
        modelOutput="",
        createdAt=created_at
    )
    
    # Generate model output
    prompt = f"{training_meta_data.plotGenerationReversePrompt}\n\n{first_draft}"
    plot_pair.reverseOutputPrompt = prompt  # Save the actual prompt sent to LLM
    response_id = llm_provider.initiateResponse(
        prompt,
        model="gpt-4.1-2025-04-14",
        resoning_effort="medium"
    )
    response = llm_provider.getPooledResponse(response_id)
    plot_pair.modelOutput = extract_response_text(response)
    generated_outputs["PLOT_GEN"] = plot_pair.modelOutput
    
    # Generate prompt for plot crime theme extraction
    theme_extraction_prompt = training_meta_data.plotCrimeThemeExtractionPrompt
    prompt = f"{theme_extraction_prompt}\n\n{first_draft}"
    
    response_id = llm_provider.initiateResponse(
        prompt,
        model="gpt-4.1-2025-04-14",
        resoning_effort="medium"
    )
    crime_theme_response = llm_provider.getPooledResponse(response_id)
    crime_theme = extract_response_text(crime_theme_response)
    plot_pair.inputPromptForFineTuning = storyverse_meta_data.plotGenerationPromptTemplate.promptTemplate.replace(
        "{{ORIGINAL_TITLE_EXAMPLES}}", title_examples
    ).replace(
        "{{CHARACTER_DETAILS}}", generated_outputs["CHARACTER_DATA_GEN"]
    ).replace(
        "{{CRIME_THEME}}", crime_theme
    )    
    
    # Save to DB
    plot_id = db_manager.create_training_io_pair(plot_pair)
    created_pairs.append(("PLOT_GEN", plot_id))
    print(f"    ✓ Saved PLOT_GEN pair with ID: {plot_id}")
    
    # STEP 3: STORY_CHAIN_GEN
    print("  - Generating STORY_CHAIN_GEN pair...")
    chain_pair = TrainingIOPair(
        storyVerse=story_verse,
        storyTitle=story_title,
        pipelineStepName="STORY_CHAIN_GEN",
        systemPrompt=training_meta_data.storyVerseSystemPrompt,
        reverseOutputPrompt="",
        inputPromptForFineTuning="",
        modelOutput="",
        createdAt=created_at
    )
    
    # Generate model output
    prompt = f"{training_meta_data.storyChainGenerationReversePrompt}\n\n{first_draft}"
    chain_pair.reverseOutputPrompt = prompt  # Save the actual prompt sent to LLM
    response_id = llm_provider.initiateResponse(
        prompt,
        model="gpt-4.1-2025-04-14",
        resoning_effort="medium"
    )
    response = llm_provider.getPooledResponse(response_id)
    chain_pair.modelOutput = extract_response_text(response)
    generated_outputs["STORY_CHAIN_GEN"] = chain_pair.modelOutput
    
    # Generate input prompt with CHARACTER_DATA and PLOT replacements
    prompt_template = storyverse_meta_data.storyChainGenerationPromptTemplate.promptTemplate
    chain_pair.inputPromptForFineTuning = prompt_template.replace(
        "{{CHARACTER_DATA}}", generated_outputs["CHARACTER_DATA_GEN"]
    ).replace(
        "{{PLOT}}", generated_outputs["PLOT_GEN"]
    )
    
    # Save to DB
    chain_id = db_manager.create_training_io_pair(chain_pair)
    created_pairs.append(("STORY_CHAIN_GEN", chain_id))
    print(f"    ✓ Saved STORY_CHAIN_GEN pair with ID: {chain_id}")
    
    # STEP 4: STORY_SUMMARY_GEN
    print("  - Generating STORY_SUMMARY_GEN pair...")
    summary_pair = TrainingIOPair(
        storyVerse=story_verse,
        storyTitle=story_title,
        pipelineStepName="STORY_SUMMARY_GEN",
        systemPrompt=training_meta_data.storyVerseSystemPrompt,
        reverseOutputPrompt="",
        inputPromptForFineTuning="",
        modelOutput="",
        createdAt=created_at
    )
    
    # Generate model output
    prompt = f"{training_meta_data.storySummaryGenerationReversePrompt}\n\n{first_draft}"
    summary_pair.reverseOutputPrompt = prompt  # Save the actual prompt sent to LLM
    response_id = llm_provider.initiateResponse(
        prompt,
        model="gpt-4.1-2025-04-14",
        resoning_effort="medium"
    )
    response = llm_provider.getPooledResponse(response_id)
    summary_pair.modelOutput = extract_response_text(response)
    generated_outputs["STORY_SUMMARY_GEN"] = summary_pair.modelOutput
    
    # Generate input prompt with CHARACTER_DATA, PLOT, and STORY_CHAIN replacements
    prompt_template = storyverse_meta_data.storySummaryGenerationPromptTemplate.promptTemplate
    summary_pair.inputPromptForFineTuning = prompt_template.replace(
        "{{CHARACTER_DATA}}", generated_outputs["CHARACTER_DATA_GEN"]
    ).replace(
        "{{STORY_PLOT}}", generated_outputs["PLOT_GEN"]
    ).replace(
        "{{STORY_CHAIN}}", generated_outputs["STORY_CHAIN_GEN"]
    )
    
    # Save to DB
    summary_id = db_manager.create_training_io_pair(summary_pair)
    created_pairs.append(("STORY_SUMMARY_GEN", summary_id))
    print(f"    ✓ Saved STORY_SUMMARY_GEN pair with ID: {summary_id}")
    
    # STEP 5: FIRST_DRAFT_GEN
    print("  - Generating FIRST_DRAFT_GEN pair...")
    draft_pair = TrainingIOPair(
        storyVerse=story_verse,
        storyTitle=story_title,
        pipelineStepName="FIRST_DRAFT_GEN",
        systemPrompt=training_meta_data.storyVerseSystemPrompt,
        reverseOutputPrompt="",
        inputPromptForFineTuning="",
        modelOutput="",
        createdAt=created_at
    )
    
    # The first draft itself is the model output - no need to reverse-engineer it
    draft_pair.modelOutput = first_draft
    generated_outputs["FIRST_DRAFT_GEN"] = draft_pair.modelOutput
    
    # For FIRST_DRAFT_GEN, reverseOutputPrompt is not applicable since we don't reverse-engineer
    # We can store the template for reference or leave it empty
    draft_pair.reverseOutputPrompt = training_meta_data.fistDraftGenerationReversePrompt
    
    # Generate input prompt with CHARACTER_DATA and STORY_SUMMARY replacements
    prompt_template = storyverse_meta_data.fistDraftGenerationPromptTemplate.promptTemplate
    draft_pair.inputPromptForFineTuning = prompt_template.replace(
        "{{CHARACTER_DATA}}", generated_outputs["CHARACTER_DATA_GEN"]
    ).replace(
        "{{STORY_SUMMARY}}", generated_outputs["STORY_SUMMARY_GEN"]
    )
    
    # Save to DB
    draft_id = db_manager.create_training_io_pair(draft_pair)
    created_pairs.append(("FIRST_DRAFT_GEN", draft_id))
    print(f"    ✓ Saved FIRST_DRAFT_GEN pair with ID: {draft_id}")
    
    return {
        "file": first_draft_file_path,
        "pairs": created_pairs
    }

In [ ]:
if __name__ == "__main__":
    # Initialize providers
    llm_provider = OpenAiLLMProvider()
    db_manager = DatabaseManager()
    
    # Get story verse from environment
    story_verse = os.getenv('STORY_VERSE', 'SHERLOCK')
    
    print(f"=== Training Data Generation Pipeline for {story_verse} ===")
    print("\n1. Fetching metadata from database...")
    
    # Fetch TrainingDataGenMetaData
    training_meta_data = db_manager.get_training_data_gen_meta_data(story_verse)
    if not training_meta_data:
        raise ValueError(f"No TrainingDataGenMetaData found for storyVerse: {story_verse}")
    print(f"   ✓ Found TrainingDataGenMetaData for {story_verse}")
    
    # Fetch StoryverseMetaData
    storyverse_meta_data = db_manager.get_meta_data(story_verse)
    if not storyverse_meta_data:
        raise ValueError(f"No StoryverseMetaData found for storyVerse: {story_verse}")
    print(f"   ✓ Found StoryverseMetaData for {story_verse}")
    
    # Get all text files from first-drafts folder
    first_drafts_dir = f"training-data/{story_verse}/first-drafts"
    text_files = glob.glob(os.path.join(first_drafts_dir, "*.txt"))
    
    if not text_files:
        print(f"\n⚠ No text files found in {first_drafts_dir}")
        print("Please add first draft stories to this directory and run again.")
    else:
        print(f"\n2. Found {len(text_files)} first draft files to process")
        for f in text_files:
            print(f"   - {os.path.basename(f)}")
        
        print(f"\n3. Processing files with thread pool (max 3 concurrent threads)...")
        
        results = []
        errors = []
        
        # Use ThreadPoolExecutor for parallel processing
        with ThreadPoolExecutor(max_workers=2) as executor:
            # Submit all tasks
            future_to_file = {
                executor.submit(
                    generateTrainingIOPair,
                    file_path,
                    training_meta_data,
                    storyverse_meta_data,
                    llm_provider,
                    db_manager,
                    story_verse
                ): file_path for file_path in text_files
            }
            
            # Collect results as they complete
            for future in as_completed(future_to_file):
                file_path = future_to_file[future]
                try:
                    result = future.result()
                    results.append(result)
                except Exception as exc:
                    error_msg = f"Error processing {file_path}: {exc}"
                    print(f"\n❌ {error_msg}")
                    errors.append(error_msg)
        
        # Print summary
        print(f"\n{'='*60}")
        print(f"Pipeline Complete!")
        print(f"{'='*60}")
        print(f"Successfully processed: {len(results)}/{len(text_files)} files")
        print(f"Total training pairs created: {len(results) * 5}")
        
        if errors:
            print(f"\n⚠ Errors encountered: {len(errors)}")
            for error in errors:
                print(f"   - {error}")
        
        if results:
            print(f"\n✓ Successfully generated training pairs for:")
            for result in results:
                print(f"   {os.path.basename(result['file'])}:")
                for step_name, pair_id in result['pairs']:
                    print(f"      - {step_name}: {pair_id}")
    
    # Close DB connection
    db_manager.close()
    print("\n✓ Database connection closed")